# Pandas Performance Profiler
### Beginner-friendly BCA / Data Science mini project

**Project goal:** Compare two ways of calculating the total value of shopping records:
1. Row-wise processing with `DataFrame.apply()`
2. Column-wise vectorized processing with Pandas

The notebook measures execution time, checks that both methods produce the same answers, measures result memory, and plots the timing comparison.

> **Note:** Timings depend on your computer and may vary between runs. This is a learning benchmark, not a universal speed test.


## 1. What you will learn

- **Pandas:** A Python library for working with tabular data.
- **DataFrame:** A table with rows and columns.
- **Performance profiling:** Measuring how efficiently code runs.
- **Execution time:** How long an operation takes.
- **Vectorization:** Applying an operation to a whole column/array rather than handling rows one at a time.
- **Memory usage:** How much memory the resulting data structure occupies.

## 2. Import the libraries

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)

## 3. Create a sample shopping dataset

We create 50,000 shopping records with a product price, quantity, and product name. A fixed random seed makes the sample data reproducible.

In [ ]:
np.random.seed(42)

NUMBER_OF_ROWS = 50_000

df = pd.DataFrame({
    "Price": np.random.randint(100, 1000, NUMBER_OF_ROWS),
    "Quantity": np.random.randint(1, 10, NUMBER_OF_ROWS),
    "Product": np.random.choice(
        ["Laptop", "Mobile", "Tablet", "Headphones"],
        NUMBER_OF_ROWS
    )
})

print("Dataset shape:", df.shape)
display(df.head())

### Understand the dataset

- `Price`: price of one unit of a product.
- `Quantity`: number of units purchased.
- `Product`: product category.
- Each row represents one shopping record.

We will calculate **Total = Price × Quantity** for every row.

## 4. Define the two methods

### Method A — Row-wise `apply()`
`apply(..., axis=1)` calls a function for each row. It is convenient, but often slower for simple arithmetic.

### Method B — Vectorized calculation
`data["Price"] * data["Quantity"]` multiplies the two columns directly. Pandas/NumPy can perform this operation efficiently over the whole column.

In [ ]:
def slow_method(data):
    result = data.copy()
    result["Total"] = result.apply(
        lambda row: row["Price"] * row["Quantity"],
        axis=1
    )
    return result


def fast_method(data):
    result = data.copy()
    result["Total"] = result["Price"] * result["Quantity"]
    return result

## 5. Create the performance profiler

The profiler:
1. Runs a method several times.
2. Measures each run with `time.perf_counter()`.
3. Calculates the average execution time.
4. Measures the memory occupied by the returned DataFrame.

**Important:** The memory figure is the size of the returned DataFrame, not the total RAM used by Python or the computer. The timing includes copying the input DataFrame, so both methods are measured on a similar basis.

In [ ]:
def profile_function(function, data, repeats=5):
    run_times = []
    result = None

    for _ in range(repeats):
        start = time.perf_counter()
        result = function(data)
        end = time.perf_counter()
        run_times.append(end - start)

    average_time = float(np.mean(run_times))
    memory_bytes = int(result.memory_usage(index=True, deep=True).sum())

    return {
        "method": function.__name__,
        "average_seconds": average_time,
        "memory_bytes": memory_bytes,
        "all_run_times": run_times,
        "result": result
    }

## 6. Run the profiler

We run each method five times and calculate the average. Multiple runs help reduce the effect of one unusually fast or slow run.

In [ ]:
slow_profile = profile_function(slow_method, df, repeats=5)
fast_profile = profile_function(fast_method, df, repeats=5)

print("SLOW METHOD (apply)")
print("Average execution time:", round(slow_profile["average_seconds"], 6), "seconds")
print("Result DataFrame memory:", f'{slow_profile["memory_bytes"]:,}', "bytes")

print("\nFAST METHOD (vectorization)")
print("Average execution time:", round(fast_profile["average_seconds"], 6), "seconds")
print("Result DataFrame memory:", f'{fast_profile["memory_bytes"]:,}', "bytes")

## 7. Check that both methods give the same answer

A fair performance comparison must compare methods that do the same task. We use `assert` to check that the calculated Total columns match.

In [ ]:
slow_result = slow_profile["result"]
fast_result = fast_profile["result"]

assert slow_result["Total"].equals(fast_result["Total"])
print("Validation successful: both methods produced identical Total values.")

display(fast_result.head())

## 8. Compare performance and draw a chart

The speed-up is calculated as:

**Speed-up = Average time of slow method ÷ Average time of fast method**

A value greater than 1 means the vectorized method was faster in this run. Your measured result may differ.

In [ ]:
slow_time = slow_profile["average_seconds"]
fast_time = fast_profile["average_seconds"]

if fast_time > 0:
    speed_up = slow_time / fast_time
else:
    speed_up = float("inf")

print(f"Apply average time:        {slow_time:.6f} seconds")
print(f"Vectorized average time:   {fast_time:.6f} seconds")
print(f"Speed-up (slow / fast):    {speed_up:.2f}x")

labels = ["Row-wise apply", "Vectorized"]
times = [slow_time, fast_time]

plt.figure(figsize=(8, 5))
bars = plt.bar(labels, times)
plt.title("Pandas Performance Comparison")
plt.ylabel("Average execution time (seconds)")
plt.xlabel("Method")
plt.bar_label(bars, fmt="%.4f")
plt.tight_layout()
plt.show()

## 9. Memory comparison

Both methods return a copy of the same input data plus one calculated column, so their result sizes should be similar. This illustrates why memory and speed are separate performance measures.

In [ ]:
memory_comparison = pd.DataFrame({
    "Method": ["Row-wise apply", "Vectorized"],
    "Result memory (bytes)": [
        slow_profile["memory_bytes"],
        fast_profile["memory_bytes"]
    ]
})

display(memory_comparison)

## 10. Conclusion

In this experiment, we calculated the total shopping value using two Pandas techniques.

- The row-wise `apply()` method processes records one by one.
- The vectorized method performs multiplication directly on two columns.
- The profiler measures average execution time and result DataFrame memory.
- The validation step confirms that both methods calculate the same values.

**Main learning:** For simple column calculations, vectorized Pandas operations are generally faster and are usually the preferred approach.

### Limitations
- Results depend on hardware, software versions, dataset size, and system activity.
- Small datasets may show a smaller difference.
- The memory measurement covers the returned DataFrame only; it does not measure total process RAM.
- This notebook compares two specific implementations, not every possible Pandas technique.

## 11. Viva questions

**1. What is Pandas?**  
Pandas is a Python library used to organize, clean, manipulate, and analyze data.

**2. What is a DataFrame?**  
A DataFrame is a two-dimensional table containing rows and columns.

**3. What is performance profiling?**  
It is the process of measuring a program's performance, such as its execution time and memory use.

**4. Why is `time.perf_counter()` used?**  
It is a high-resolution timer suitable for measuring elapsed time.

**5. What does `axis=1` mean in `apply()`?**  
It tells Pandas to apply the function row by row.

**6. What is vectorization?**  
It means applying an operation to an entire array or column instead of explicitly processing each row in Python.

**7. Why do we repeat the measurement?**  
To calculate an average and reduce the influence of an unusually fast or slow run.

**8. Why do we validate the results?**  
To make sure both methods perform the same calculation before comparing their speed.

**9. Does this project measure total computer RAM?**  
No. It measures the memory occupied by the returned DataFrame.

**10. What is the main conclusion?**  
Vectorized operations are generally faster for simple column calculations in Pandas.

---

### How to run this notebook
1. Open Google Colab: https://colab.research.google.com/
2. Choose **File → Upload notebook**.
3. Upload this `.ipynb` file.
4. Choose **Runtime → Run all**.
5. Review the output and chart, then save a copy to your Google Drive.
